# Gemma 4 31B Instruct — Kaggle TPU v5e-8 — G0-G2 Corrective R1

Upload the **R1 full source ZIP** by SSH to `/kaggle/working`, attach the Keras model containing `gemma4_instruct_31b`, enable Internet, and select TPU v5e-8. R1 changes only checkpoint-loading semantics and still stops after G2 strict-load evidence.


In [ ]:
from pathlib import Path
import zipfile, shutil
WORK=Path('/kaggle/working/gemma4-project')
if WORK.exists(): shutil.rmtree(WORK)
candidates=[]
candidates.extend(Path('/kaggle/working').glob('KerasHub-Gemma4-31B-IT-Kaggle-TPU-v5e8-Text-Vision-source-r1.zip'))
if not candidates:
    candidates=[p for p in Path('/kaggle/working').glob('*.zip') if 'Gemma4-31B' in p.name and 'source' in p.name]
if not candidates: raise FileNotFoundError('SSH upload the R1 full source ZIP to /kaggle/working')
extract=Path('/kaggle/working/gemma4-source-extract')
if extract.exists(): shutil.rmtree(extract)
extract.mkdir()
with zipfile.ZipFile(candidates[0]) as z: z.extractall(extract)
roots=[p for p in extract.iterdir() if p.is_dir()]
srcroot=roots[0] if len(roots)==1 else extract
shutil.copytree(srcroot,WORK)
print('SOURCE=',candidates[0])
print('WORK=',WORK)


In [ ]:
import subprocess
subprocess.run(['bash','-lc','cp -f .env.example .env && bash scripts/run_g0_g2.sh'],cwd=WORK,check=True)


In [ ]:
import json
result=json.loads((WORK/'artifacts/g0-g2/strict-load.json').read_text())
print(json.dumps(result,indent=2,sort_keys=True))
assert result['status']=='strict_load_pass'
assert result['device_count']==8
assert result['mesh_shape']==[1,8]
assert result['model_class']=='Gemma4CausalLM'
assert result['num_layers']==60
assert result['generation_executed'] is False
assert result['checkpoint_load_strategy']=='keras_hub_native_preset_loader'
assert result['checkpoint_target']=='task.backbone'
assert result['skip_mismatch'] is False
print('G2_STRICT_LOAD_PASS')


## First-run stop point

Send `artifacts/g0-g2/strict-load.json` for adjudication before spending TPU quota on G3/G5 native generation.
